In [1]:
import sys
import os
 
# Add project root to Python path
sys.path.append(os.path.abspath(".."))

In [17]:
from src.bayesian_network.label_data import label_dataset

raw_path = "../data/raw/bank_transactions_data_2.csv"
anomaly_path = "../results/dbscan/dbscan_anomaly_table.csv"
output_path = "../data/processed/bank_transactions_labeled.csv"

df_labeled = label_dataset(raw_path, anomaly_path, output_path)

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/bank_transactions_labeled.csv")
df.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,anomaly
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08,0
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35,0
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04,0
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06,0
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39,0


In [3]:
from src.bayesian_network.preprocess_bn import discretize_dataset

df_disc = discretize_dataset(df)
df_disc.head()


Numeric columns being discretized: ['TransactionAmount', 'CustomerAge', 'TransactionDuration', 'LoginAttempts', 'AccountBalance']


c:\Users\tewod\fraud-detection-unsupervised-learning\venv\lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\tewod\fraud-detection-unsupervised-learning\venv\lib\site-packages\sklearn\preprocessing\_discretization.py:397: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.
  warnings.warn(


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,anomaly
0,TX000001,AC00128,0.0,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,3.0,Doctor,1.0,0.0,2.0,2024-11-04 08:08:08,0
1,TX000002,AC00455,2.0,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,3.0,Doctor,2.0,0.0,3.0,2024-11-04 08:09:35,0
2,TX000003,AC00019,1.0,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,0.0,Student,0.0,0.0,0.0,2024-11-04 08:07:04,0
3,TX000004,AC00070,1.0,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,0.0,Student,0.0,0.0,3.0,2024-11-04 08:09:06,0
4,TX000005,AC00411,0.0,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,0.0,Student,3.0,0.0,2.0,2024-11-04 08:06:39,0


In [4]:
from src.bayesian_network.structure_learning import learn_structure

model_structure = learn_structure(df_disc)
model_structure.edges()


c:\Users\tewod\fraud-detection-unsupervised-learning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 3/1000000 [00:01<112:25:33,  2.47it/s]


OutEdgeView([('TransactionType', 'Channel'), ('CustomerAge', 'CustomerOccupation'), ('CustomerOccupation', 'AccountBalance')])

In [5]:
from src.bayesian_network.structure_learning import learn_structure
from src.bayesian_network.cpd_learning import learn_cpds

model_structure = learn_structure(df_disc)
model = learn_cpds(model_structure, df_disc)

model.get_cpds()

  0%|          | 3/1000000 [00:01<103:56:48,  2.67it/s]


[<TabularCPD representing P(TransactionType:2) at 0x28c49feeb90>,
 <TabularCPD representing P(Channel:3 | TransactionType:2) at 0x28c49feeb00>,
 <TabularCPD representing P(CustomerAge:4) at 0x28c49fee9e0>,
 <TabularCPD representing P(CustomerOccupation:4 | CustomerAge:4) at 0x28c49feea70>,
 <TabularCPD representing P(AccountBalance:4 | CustomerOccupation:4) at 0x28c49fed060>]